In [1]:
import sys
print(sys.version)
print(sys.executable)

3.11.18 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 18:13:29) [MSC v.1942 64 bit (AMD64)]
C:\Users\jiffy\anaconda3\envs\Interviewcoach\python.exe


In [2]:
%pip install sentence-transformers faiss-cpu pypdf numpy pandas requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import pypdf
from pypdf import PdfReader
import numpy as np
import pandas as pd
import requests

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [2]:
from pathlib import Path

project_folder = Path(
    r"C:\Users\jiffy\OneDrive\Documents\AI_Interview_Coach"
)

print("Project folder exists:", project_folder.exists())

for folder in ["knowledge_base", "resume", "app"]:
    path = project_folder / folder

    print(f"\n{folder}: {'Found' if path.exists() else 'Not found'}")

    if path.exists():
        for file in path.iterdir():
            print("  -", file.name)

Project folder exists: True

knowledge_base: Found
  - ai_ml_engineer_interview_guide.pdf
  - data_scientist_interview_guide.pdf
  - hr_behavioral_interview_questions.pdf
  - hr_interview_questions_and_sample_answers.pdf
  - ml_interview_knowledge_base.pdf
  - resume_project_deep_dive_guide.pdf

resume: Found
  - Jiffy_Jacob.pdf

app: Found
  - app.py
  - rag_engine.py
  - __pycache__


In [3]:
from pathlib import Path

knowledge_folder = project_folder / "knowledge_base"
resume_folder = project_folder / "resume"

knowledge_pdfs = list(knowledge_folder.glob("*.pdf"))
resume_pdfs = list(resume_folder.glob("*.pdf"))

print("Knowledge-base PDFs:", len(knowledge_pdfs))

for pdf in knowledge_pdfs:
    print("-", pdf.name)

print("\nResume PDFs:", len(resume_pdfs))

for pdf in resume_pdfs:
    print("-", pdf.name)

Knowledge-base PDFs: 6
- ai_ml_engineer_interview_guide.pdf
- data_scientist_interview_guide.pdf
- hr_behavioral_interview_questions.pdf
- hr_interview_questions_and_sample_answers.pdf
- ml_interview_knowledge_base.pdf
- resume_project_deep_dive_guide.pdf

Resume PDFs: 1
- Jiffy_Jacob.pdf


**Extract Text from pdfs**

In [4]:
from pathlib import Path
from pypdf import PdfReader

# Project folders
project_folder = Path(
    r"C:\Users\jiffy\OneDrive\Documents\AI_Interview_Coach"
)

knowledge_folder = project_folder / "knowledge_base"
resume_folder = project_folder / "resume"

# Function to extract text from a PDF
def extract_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )
    return text.strip()

# Extract knowledge-base text
knowledge_documents = {}

for pdf_path in knowledge_folder.glob("*.pdf"):
    text = extract_pdf_text(pdf_path)
    knowledge_documents[pdf_path.name] = text
    print(f"{pdf_path.name}: {len(text)} characters")

# Extract resume text
resume_documents = {}

for pdf_path in resume_folder.glob("*.pdf"):
    text = extract_pdf_text(pdf_path)
    resume_documents[pdf_path.name] = text
    print(f"{pdf_path.name}: {len(text)} characters")

print("\nPDF text extraction completed.")


ai_ml_engineer_interview_guide.pdf: 6668 characters
data_scientist_interview_guide.pdf: 7159 characters
hr_behavioral_interview_questions.pdf: 6326 characters
hr_interview_questions_and_sample_answers.pdf: 8051 characters
ml_interview_knowledge_base.pdf: 18346 characters
resume_project_deep_dive_guide.pdf: 6026 characters
Jiffy_Jacob.pdf: 2155 characters

PDF text extraction completed.


**Split Extracted Text into Chunks**

In [5]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []

    if not text:
        return chunks

    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


# Create chunks from knowledge-base PDFs
knowledge_chunks = []

for filename, text in knowledge_documents.items():
    chunks = split_text(text)

    for chunk in chunks:
        knowledge_chunks.append({
            "source": filename,
            "text": chunk
        })


# Create chunks from the resume PDFs
resume_chunks = []

for filename, text in resume_documents.items():
    chunks = split_text(text)

    for chunk in chunks:
        resume_chunks.append({
            "source": filename,
            "text": chunk
        })


print("Knowledge-base chunks:", len(knowledge_chunks))
print("Resume chunks:", len(resume_chunks))

# Preview one knowledge-base chunk
if knowledge_chunks:
    print("\nSample knowledge-base chunk:")
    print(knowledge_chunks[0]["text"])


Knowledge-base chunks: 132
Resume chunks: 6

Sample knowledge-base chunk:
Resume-Based Interview Prep | Original study guide
1
AI / ML Engineer Interview Guide
Original practice prompts and concise answer frameworks
A production-oriented guide across model development, evaluation, deployment, monitoring, ML systems, and
LLM applications. Designed for retrieval by a resume-based interview preparation app.
How to use this guide
Treat each item as a practice prompt, not a script. Answer aloud, anchor claims in your own resume, and adapt
the depth to the role and seniorit


**Generate MiniLM Embeddings**

In [6]:
#Load MiniLM
from sentence_transformers import SentenceTransformer

# Load the pretrained embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("MiniLM model loaded successfully!")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7241.86it/s]


MiniLM model loaded successfully!


**Generate embeddings and Build FAISS index**

In [7]:
import numpy as np
import faiss

# Extract text from knowledge-base chunks
knowledge_texts = [
    chunk["text"] for chunk in knowledge_chunks
]

# Generate embeddings
knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert embeddings to float32 for FAISS
knowledge_embeddings = knowledge_embeddings.astype("float32")

# Build the FAISS index
embedding_dimension = knowledge_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(knowledge_embeddings)

print("Embedding shape:", knowledge_embeddings.shape)
print("FAISS index created successfully!")
print("Total indexed chunks:", faiss_index.ntotal)


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.09s/it]

Embedding shape: (132, 384)
FAISS index created successfully!
Total indexed chunks: 132


In [9]:
#Test FAISS Retrieval
def retrieve_context(query, top_k=3):
    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Search the FAISS index
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for score, index in zip(scores[0], indices[0]):
        if index != -1:
            chunk = knowledge_chunks[index]

            results.append({
                "score": float(score),
                "source": chunk["source"],
                "text": chunk["text"]
            })

    return results


# Test retrieval
query = "Explain Random Forest in simple terms"

results = retrieve_context(query)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Similarity score:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Text:", result["text"][:500])


--- Result 1 ---
Similarity score: 0.5152
Source: ml_interview_knowledge_base.pdf
Text: ch as 0.5 turns that into
a predicted cancellation.
Likely follow-ups How would you choose a threshold? / Why might the default threshold be unsuitable?
5. How does a decision tree work, and what is a Random Forest?
Answer framework A decision tree repeatedly splits examples using feature rules to make groups more predictive of the
target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree m

--- Result 2 ---
Similarity score: 0.399
Source: ml_interview_knowledge_base.pdf
Text: les and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6.

In [10]:
query = "What is the difference between overfitting and underfitting?"

results = retrieve_context(query, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Similarity score:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Text:", result["text"][:400])


--- Result 1 ---
Similarity score: 0.5302
Source: ml_interview_knowledge_base.pdf
Text: erage in absolute
terms.
Likely follow-ups Which metric is more sensitive to outliers? / Can R-squared be negative on held-out data?
3. What are overfitting and underfitting?
Answer framework Overfitting means a model learns training-specific noise and performs worse on new data. Underfitting
means it is too simple or poorly trained to capture useful patterns, so it performs poorly even on trainin

--- Result 2 ---
Similarity score: 0.3734
Source: ml_interview_knowledge_base.pdf
Text: les and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6. When might you use KNN or an SVM?
Answer framework K-nearest neighbors predic

--- Result 3 ---
Si

**Connect Qwen to RAG system**

In [11]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen2.5:7b"

print("Qwen connection configured successfully!")

Qwen connection configured successfully!


**Create the RAG answer generation function**

In [12]:
def generate_rag_answer(question, top_k=3):

    # Step 1: Retrieve relevant information from FAISS
    retrieved_results = retrieve_context(
        question,
        top_k=top_k
    )

    if not retrieved_results:
        return (
            "No relevant information was retrieved "
            "from the knowledge base."
        )

    # Step 2: Combine retrieved information
    context_parts = []

    for item in retrieved_results:

        if isinstance(item, dict):

            source = item.get("source", "Knowledge Base")
            text = item.get("text", "")

            if text.strip():
                context_parts.append(
                    f"Source: {source}\n{text}"
                )

        else:

            text = str(item)

            if text.strip():
                context_parts.append(text)

    context = "\n\n".join(context_parts)

    # Step 3: Create the prompt for Qwen
    prompt = f"""
You are an experienced Data Science and AI/ML interview coach.

Answer the following interview question using the retrieved context.

INTERVIEW QUESTION:
{question}

RETRIEVED KNOWLEDGE:
{context}

INSTRUCTIONS:
- Explain the concept in simple, clear English.
- Keep the answer concise, usually 2-5 sentences.
- Give a short example when useful.
- Be technically accurate.
- Avoid unnecessary advanced details.
- Do not invent facts unsupported by the context.
- If the context is insufficient, acknowledge the limitation.
- Focus directly on answering the question.

Provide the answer directly without repeating the question.
"""

    # Step 4: Send the prompt to Qwen through Ollama
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0.1
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    # Step 5: Extract the generated answer
    result = response.json()

    if "message" not in result:
        raise ValueError(
            f"Unexpected Ollama response: {result}"
        )

    answer = result["message"].get("content", "").strip()

    if not answer:
        raise ValueError(
            "Qwen returned an empty answer."
        )

    return answer


In [14]:
question = "Explain Random Forest in simple terms."

answer = generate_rag_answer(question)

print("QUESTION:")
print(question)

print("\nQWEN'S RAG-BASED ANSWER:")
print(answer)

QUESTION:
Explain Random Forest in simple terms.

QWEN'S RAG-BASED ANSWER:
Random Forest is a machine learning algorithm that combines multiple decision trees to make predictions. Each decision tree is trained on a random subset of the data and features, which helps reduce the instability of individual trees. The final prediction is made by averaging the predictions of all the trees. This ensemble approach typically leads to more stable and accurate predictions compared to a single decision tree.

For example, imagine predicting whether a customer will return a product. A Random Forest might split customers based on different criteria like prior returns and order value, and then combine these splits to make a more reliable prediction.


In [13]:
question = "What information does the knowledge base provide about Random Forest?"

results = retrieve_context(question, top_k=3)

print("RETRIEVED KNOWLEDGE-BASE CONTEXT\n")

for i, item in enumerate(results, start=1):
    print(f"Result {i}")
    print(f"Source: {item['source']}")
    print(f"Similarity score: {item['score']:.4f}")
    print(item["text"])
    print("-" * 60)
    

RETRIEVED KNOWLEDGE-BASE CONTEXT

Result 1
Source: ml_interview_knowledge_base.pdf
Similarity score: 0.4409
ch as 0.5 turns that into
a predicted cancellation.
Likely follow-ups How would you choose a threshold? / Why might the default threshold be unsuitable?
5. How does a decision tree work, and what is a Random Forest?
Answer framework A decision tree repeatedly splits examples using feature rules to make groups more predictive of the
target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree m
------------------------------------------------------------
Result 2
Source: ml_interview_knowledge_base.pdf
Similarity score: 0.4098
les and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can 

In [17]:
question = "What is Random Forest, and how does it reduce overfitting?"

answer = generate_rag_answer(question, top_k=3)

print("INTERVIEW QUESTION")
print(question)

print("\nGENERATED ANSWER")
print(answer)

INTERVIEW QUESTION
What is Random Forest, and how does it reduce overfitting?

GENERATED ANSWER
Random Forest is an ensemble method that combines multiple decision trees to make predictions. It reduces overfitting by averaging the predictions of many trees, each trained on a different subset of the data and features. This averaging process makes the model less sensitive to the specific data it was trained on, leading to more stable and generalizable results.

For example, if you have a dataset with 1000 samples, a Random Forest might train 100 decision trees, each on a random subset of 500 samples and features. The final prediction is the average of the predictions from these 100 trees, which helps to smooth out any noise or outliers in the data.


**RAG Based Interview Answer Evaluator**

In [18]:
import json
def evaluate_interview_answer(question, candidate_answer, top_k=3):

    # --------------------------------------------------
    # 1. RETRIEVE RELEVANT KNOWLEDGE
    # --------------------------------------------------

    retrieved_results = retrieve_context(
        question,
        top_k=top_k
    )

    context_parts = []

    for item in retrieved_results:

        if isinstance(item, dict):
            text = item.get("text", "")
        else:
            text = str(item)

        if text.strip():
            context_parts.append(text)

    context = "\n\n".join(context_parts)

    # --------------------------------------------------
    # 2. CREATE THE EVALUATION PROMPT
    # --------------------------------------------------

    prompt = f"""
You are a fair, supportive, and knowledgeable interviewer
conducting a mock interview for a fresher applying for a
Data Science or AI/ML position.

Evaluate the candidate's actual answer carefully.

QUESTION:
{question}

REFERENCE KNOWLEDGE:
{context}

CANDIDATE'S ANSWER:
{candidate_answer}

==================================================
SCORING RUBRIC: TOTAL 10 MARKS
==================================================

1. Technical correctness: 0-4 marks
   Are the statements factually and technically correct?

2. Conceptual understanding: 0-3 marks
   Does the candidate demonstrate an understanding of the concept?

3. Completeness: 0-2 marks
   Does the answer cover the essential points required by this question?

4. Clarity: 0-1 mark
   Is the answer understandable?

The maximum total is 10 marks.

==================================================
IMPORTANT FAIRNESS RULES
==================================================

1. Evaluate what the candidate actually said, not what you
   expected them to say.

2. Accept correct explanations expressed in simple language.
   Advanced terminology is not compulsory when the meaning is correct.

3. A short answer can receive a high score if it correctly covers
   the essential points of the question.

4. Do not demand examples, formulas, advantages, disadvantages,
   implementation details, or advanced concepts unless they are
   relevant to the question.

5. Give credit for equivalent explanations and correct synonyms.

6. Do not mark a correct statement as incorrect.

7. Do not list a concept as missing if the candidate has already
   explained it correctly.

8. Keep these categories separate:
   - strengths: correctly explained concepts
   - missing_concepts: important concepts not explained adequately
   - incorrect_statements: genuinely false claims made by the candidate

9. Do not penalize the candidate twice for the same omission.

10. Do not lower the score simply because the answer could be longer.

11. Calculate the total score by adding the four component scores.

12. Make the score consistent with the rubric and your written feedback.

==================================================
IMPROVED ANSWER
==================================================

Write a better version of the candidate's answer.

- Use simple, interview-appropriate English.
- Preserve the candidate's correct points.
- Correct genuine factual errors.
- Add only essential missing information.
- Keep it concise, usually 2-5 sentences, depending on the question.
- Do not unnecessarily expand a simple definition into a long lecture.
- If the original answer is already excellent, retain its meaning and
  explain in feedback that only minor or no changes are needed.
- Do not copy the original answer word for word when a meaningful
  improvement is possible.

==================================================
ADAPTIVE FOLLOW-UP QUESTION
==================================================

Generate exactly ONE short follow-up question based on the
candidate's actual answer.

Rules:

1. The follow-up must relate directly to the original question
   and the candidate's answer.

2. If the answer is incomplete, ask about one important missing concept.

3. If the answer is correct but basic, ask a natural next question
   that explores the concept a little more deeply.

4. If the answer contains a misconception, ask a question that helps
   clarify or test the misunderstood concept without being hostile.

5. If the answer is excellent, ask a slightly more challenging
   related question.

6. Keep the follow-up concise, ideally one sentence.

7. Do not repeat the original question.

8. Do not ask several questions in one sentence.

9. Do not make the follow-up unnecessarily difficult for a fresher.

10. Do not ask unrelated questions.

11. The follow-up must be a question the interviewer can ask next.
    It must not contain its own answer.

==================================================
FEEDBACK
==================================================

Give brief, specific, encouraging feedback.

Explain what the candidate did well and the most important thing
they should improve.

Avoid generic statements that do not explain the actual answer.

==================================================
RETURN ONLY VALID JSON
==================================================

Use exactly this structure:

{{
    "score": 0,
    "score_breakdown": {{
        "technical_correctness": 0,
        "conceptual_understanding": 0,
        "completeness": 0,
        "clarity": 0
    }},
    "strengths": [],
    "missing_concepts": [],
    "incorrect_statements": [],
    "feedback": "",
    "improved_answer": "",
    "follow_up_question": ""
}}

Return valid JSON only.
Do not include Markdown code fences.
"""


    # --------------------------------------------------
    # 3. SEND THE REQUEST TO QWEN THROUGH OLLAMA
    # --------------------------------------------------

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0.1
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    result_text = response.json()["message"]["content"]

    evaluation = json.loads(result_text)


    # --------------------------------------------------
    # 4. VALIDATE AND NORMALIZE THE SCORES
    # --------------------------------------------------

    rubric_limits = {
        "technical_correctness": 4,
        "conceptual_understanding": 3,
        "completeness": 2,
        "clarity": 1
    }

    breakdown = evaluation.get("score_breakdown", {})

    validated_breakdown = {}

    for criterion, maximum in rubric_limits.items():

        try:
            value = float(breakdown.get(criterion, 0))

        except (TypeError, ValueError):
            value = 0

        validated_breakdown[criterion] = round(
            max(0, min(maximum, value)),
            1
        )

    evaluation["score_breakdown"] = validated_breakdown

    # Always calculate the total from the component scores.
    evaluation["score"] = round(
        sum(validated_breakdown.values()),
        1
    )


    # --------------------------------------------------
    # 5. ENSURE REQUIRED FIELDS EXIST
    # --------------------------------------------------

    evaluation.setdefault("strengths", [])
    evaluation.setdefault("missing_concepts", [])
    evaluation.setdefault("incorrect_statements", [])
    evaluation.setdefault("feedback", "")
    evaluation.setdefault("improved_answer", "")
    evaluation.setdefault("follow_up_question", "")

    return evaluation


In [19]:
question = "What is Random Forest, and how does it reduce overfitting?"

candidate_answer = """
Random Forest is a machine learning algorithm that uses multiple
decision trees. It combines their predictions to make a final
prediction.
"""

evaluation = evaluate_interview_answer(
    question,
    candidate_answer
)

print("QUESTION:")
print(question)

print("\nCANDIDATE'S ANSWER:")
print(candidate_answer)

print("\nEVALUATION:")
print(json.dumps(evaluation, indent=4))

QUESTION:
What is Random Forest, and how does it reduce overfitting?

CANDIDATE'S ANSWER:

Random Forest is a machine learning algorithm that uses multiple
decision trees. It combines their predictions to make a final
prediction.


EVALUATION:
{
    "score": 7.0,
    "score_breakdown": {
        "technical_correctness": 2.0,
        "conceptual_understanding": 2.0,
        "completeness": 2,
        "clarity": 1
    },
    "strengths": [
        "Candidate correctly identified Random Forest as a machine learning algorithm using multiple decision trees."
    ],
    "missing_concepts": [
        "Candidate did not explain how Random Forest reduces overfitting."
    ],
    "incorrect_statements": [],
    "feedback": "The candidate correctly identified Random Forest as a machine learning algorithm using multiple decision trees. However, the answer is incomplete as it does not explain how Random Forest reduces overfitting. To improve, the candidate should include the concept of averaging mu

In [20]:
question = "What is Random Forest, and how does it reduce overfitting?"

candidate_answer = """
Random Forest is an ensemble machine learning algorithm that combines
multiple decision trees. Each tree is trained using a random sample of
the training data, and a random subset of features is considered when
building the trees. For classification, the trees combine their
predictions using majority voting, while regression uses averaging.

A single decision tree can overfit by learning noise in the training
data. Random Forest reduces this risk by combining predictions from
many trees, reducing variance and making the model more stable on
unseen data.
"""

evaluation = evaluate_interview_answer(
    question,
    candidate_answer
)

print(json.dumps(evaluation, indent=4))

{
    "score": 10,
    "score_breakdown": {
        "technical_correctness": 4,
        "conceptual_understanding": 3,
        "completeness": 2,
        "clarity": 1
    },
    "strengths": [
        "Candidate correctly explained Random Forest as an ensemble of decision trees.",
        "Candidate mentioned the use of random samples and feature subsets to build trees."
    ],
    "missing_concepts": [
        "Candidate did not mention the majority voting or averaging process for combining predictions."
    ],
    "incorrect_statements": [],
    "feedback": "The candidate provided a good basic understanding of Random Forest. The explanation is clear and concise. To improve, it would be helpful to include the process of combining predictions (majority voting for classification or averaging for regression).",
    "improved_answer": "Random Forest is an ensemble machine learning algorithm that combines multiple decision trees. Each tree is trained using a random sample of the training d

**Question Generation**

In [21]:
def generate_interview_questions(job_role, interview_type, num_questions=5):

    # 1. Extract resume information
    resume_texts = []

    for chunk in resume_chunks:
        if isinstance(chunk, dict):
            text = chunk.get("text", "")
        else:
            text = str(chunk)

        if text.strip():
            resume_texts.append(text)

    resume_context = "\n\n".join(resume_texts)

    # 2. Retrieve relevant knowledge-base information
    if "technical" in interview_type.lower():

        query = (
            f"Short technical interview questions for {job_role}, "
            "covering Python, machine learning, deep learning, NLP, "
            "and resume-based project discussions."
        )

        top_k = 4

        instructions = """
- Ask about Python, machine learning, deep learning, NLP,
  and the candidate's actual resume projects.
- Include conceptual questions and questions about project decisions.
- Ask questions suitable for a fresher or entry-level candidate.
- Keep each question to one sentence whenever possible.
- Prefer questions that can be answered in 30-90 seconds.
- Ask only ONE main question at a time in the interview.
- Do not combine multiple questions using "and".
- Avoid long, complicated, or repetitive questions.
- Do not invent projects, skills, or experience.
"""

    else:

        query = (
            "HR interview questions about self-introduction, motivation, "
            "strengths, weaknesses, teamwork, challenges, adaptability, "
            "career goals, and handling failure."
        )

        top_k = 2

        instructions = """
- At least 4 out of 5 questions should be general HR or behavioral questions.
- Cover self-introduction, motivation, strengths, weaknesses,
  teamwork, challenges, adaptability, and career goals.
- You may ask about the candidate's personal contribution to a project.
- Do not ask technical questions about algorithms, metrics, or implementation.
- Keep questions short, natural, and conversational.
- Prefer questions that can be answered in 30-90 seconds.
- Ask only ONE thing per question.
- Avoid long, complicated, or repetitive questions.
"""

    retrieved_results = retrieve_context(query, top_k=top_k)

    knowledge_texts = []

    for item in retrieved_results:
        if isinstance(item, dict):
            text = item.get("text", "")
        else:
            text = str(item)

        if text.strip():
            knowledge_texts.append(text)

    knowledge_context = "\n\n".join(knowledge_texts)

    # 3. Build the prompt
    prompt = f"""
You are an experienced and friendly interviewer conducting
a realistic interview for a fresher.

JOB ROLE:
{job_role}

INTERVIEW TYPE:
{interview_type}

CANDIDATE RESUME:
{resume_context}

INTERVIEW KNOWLEDGE:
{knowledge_context}

TASK:
Generate exactly {num_questions} distinct interview questions.

RULES:
{instructions}

Additional rules:
- Use simple, natural spoken English.
- Do not include answers, explanations, headings, or introductions.
- Do not add follow-up questions to this list.
- Each numbered item must contain one question only.
- Make questions relevant to the candidate's resume and job role.
- Return only the numbered list.
"""

    # 4. Send the request to Qwen through Ollama
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0.7
        }
    }

    response = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    result = response.json()

    if "message" not in result:
        raise ValueError(
            f"Unexpected Ollama response: {result}"
        )

    questions_text = result["message"].get("content", "").strip()

    if not questions_text:
        raise ValueError(
            "Qwen returned an empty response. Please try again."
        )

    return questions_text


In [22]:
hr_questions = generate_interview_questions(
    job_role="AI/ML Intern",
    interview_type="HR",
    num_questions=5
)

print("HR INTERVIEW QUESTIONS")
print(hr_questions)

HR INTERVIEW QUESTIONS
1. Can you tell us a bit about yourself and how you got interested in AI and ML?
2. What motivated you to pursue an internship in AI/ML, and how do you see this role fitting into your career goals?
3. Could you describe a situation where you had to work in a team and what role did you play?
4. What challenges did you face in your previous ML projects, and how did you overcome them?
5. How do you demonstrate adaptability in your work, especially when dealing with new tools or technologies like those you mentioned in your resume?


In [23]:
# TEST: Technical Interview Question Generation

technical_questions = generate_interview_questions(
    job_role="AI/ML Intern",
    interview_type="Technical",
    num_questions=5
)

print("TECHNICAL INTERVIEW QUESTIONS")
print("=" * 50)
print(technical_questions)

TECHNICAL INTERVIEW QUESTIONS
1. Can you explain how you approached the E-Commerce sales prediction project, and what was the most challenging part of it?
2. What motivated you to choose Random Forest as one of the algorithms for the sales prediction model, and what were the results?
3. Could you describe your process for developing the book recommendation system, and which collaborative filtering approach did you find most effective?
4. How did you evaluate the performance of the emotion classification model, and what were the key metrics you used?
5. In your machine learning projects, you've worked with both supervised and unsupervised learning. Can you give an example of when you used unsupervised learning, and what was the outcome?


In [24]:
# TEST: Adaptive Follow-Up Question Generation

question = "What is Random Forest, and how does it reduce overfitting?"

candidate_answer = """
Random Forest is a machine learning algorithm that combines
multiple decision trees to make predictions.
"""

evaluation = evaluate_interview_answer(
    question,
    candidate_answer
)

print("ORIGINAL QUESTION:")
print(question)

print("\nCANDIDATE'S ANSWER:")
print(candidate_answer)

print("\nSCORE:")
print(evaluation.get("score"), "/ 10")

print("\nFEEDBACK:")
print(evaluation.get("feedback"))

print("\nMISSING CONCEPTS:")
print(evaluation.get("missing_concepts"))

print("\nFOLLOW-UP QUESTION:")
print(evaluation.get("follow_up_question"))

ORIGINAL QUESTION:
What is Random Forest, and how does it reduce overfitting?

CANDIDATE'S ANSWER:

Random Forest is a machine learning algorithm that combines
multiple decision trees to make predictions.


SCORE:
6.0 / 10

FEEDBACK:
The candidate correctly identified Random Forest as a machine learning algorithm that combines multiple decision trees. However, the answer is incomplete as it does not explain how Random Forest reduces overfitting. To improve, the candidate should include that Random Forest reduces overfitting by averaging the predictions of many trees, each trained on a different subset of the data and features, which typically leads to more stable and generalizable models.

MISSING CONCEPTS:
['Candidate did not explain how Random Forest reduces overfitting.']

FOLLOW-UP QUESTION:
Can you explain how the process of training each tree on a different subset of the data and features helps in reducing overfitting?


In [23]:
# TEST: Evaluation Output Validation

required_fields = [
    "score",
    "score_breakdown",
    "strengths",
    "missing_concepts",
    "incorrect_statements",
    "feedback",
    "improved_answer",
    "follow_up_question"
]

missing_fields = [
    field for field in required_fields
    if field not in evaluation
]

if not missing_fields:
    print("SUCCESS: All required evaluation fields are present.")
else:
    print("Missing fields:", missing_fields)

print("\nScore:", evaluation.get("score"), "/ 10")
print("Score breakdown:", evaluation.get("score_breakdown"))

SUCCESS: All required evaluation fields are present.

Score: 6.0 / 10
Score breakdown: {'technical_correctness': 3.0, 'conceptual_understanding': 2.0, 'completeness': 1.0, 'clarity': 0}
